In [ ]:
# ==========================================================
# 04 - Tree Based Models
# ==========================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from src.models.preprocessor import get_preprocessor

print("="*60)
print("Loading Dataset")
print("="*60)

df = pd.read_parquet("../data/processed/featured_taxi_data.parquet")

drop_columns = [
    "trip_duration_minutes",
    "tpep_dropoff_datetime",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "tolls_amount",
    "extra",
    "airport_fee",
    "Airport_fee",
    "congestion_surcharge",
    "improvement_surcharge"
]

X = df.drop(columns=drop_columns)
y = df["trip_duration_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

models = {
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
        max_depth=10
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )
}

results = []

best_model = None
best_mae = float("inf")

print("\nTraining Tree Models...\n")

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", get_preprocessor()),
        ("model", model)
    ])

    start = time.time()

    pipeline.fit(X_train, y_train)

    train_time = time.time() - start

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    print(name)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"Time : {train_time:.2f} sec")
    print()

    results.append([
        name,
        mae,
        rmse,
        r2,
        train_time
    ])

    if mae < best_mae:
        best_mae = mae
        best_model = pipeline

results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2",
        "Training Time"
    ]
)

print("="*60)
print("Results")
print("="*60)

print(results)

# =============================
# MAE Comparison
# =============================

plt.figure(figsize=(8,5))

plt.bar(
    results["Model"],
    results["MAE"]
)

plt.title("Tree Models MAE Comparison")

plt.ylabel("MAE")

plt.xticks(rotation=15)

plt.tight_layout()

plt.show()

# =============================
# Training Time Comparison
# =============================

plt.figure(figsize=(8,5))

plt.bar(
    results["Model"],
    results["Training Time"]
)

plt.title("Training Time Comparison")

plt.ylabel("Seconds")

plt.xticks(rotation=15)

plt.tight_layout()

plt.show()

print("\nBest Model")

print(best_model)

print("\nCompleted Successfully.")